In this notebook we'll explore different document loaders available through `langchain`.

## PyPDF loader

This is suited to simple, clean PDFs.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# doc1 contains 4 columns per page, so I'm testing whether the loader can handle this layout

doc1_path = "/Users/ashapatel/Documents/projects/rag_cc/leaflets_and_guidelines/PILs/motion_sickness_PIL.pdf"

# doc 2 is a more standard layout, with 1 column per page

doc2_path = "/Users/ashapatel/Documents/projects/rag_cc/leaflets_and_guidelines/PILs/antacid_PIL.pdf"

In [ ]:
loader1 = PyPDFLoader(doc1_path)
document1 = loader1.load()

In [ ]:
loader2 = PyPDFLoader(doc2_path)
document2 = loader2.load()

### Explore the extracted page content

In [ ]:
for page_num, page in enumerate(document1):
    print(f"=== Page {page_num + 1} ===")
    print(page.page_content)
    print()

This document loader correctly read down each column before moving to the next one. However, one interesting artifact is that at the bottom of the first column it extracts the numbers from the section headings which go across the page horizontally. This might make it difficult for the RAG system to refer to a specific section number.

In [ ]:
for page_num, page in enumerate(document2):
    print(f"=== Page {page_num + 1} ===")
    print(page.page_content)
    print()

The second document looks good. The only mistake I noticed is that the loader added a space in the middle of the word 'quantitative'. I don't think there's a fix for this, but I don't think it's too much of a problem since I imagine an LLM will understand that the space isn't meant to be there.

### Cleaning page content

In document 2, there's a lot of whitespace e.g between section headings and the text in the given section. We could remove this whitespace so that the section heading and text are more likely to appear in the same chunk, which helps preserve context and also reduces token usage.

Let's write a function to clean the page content:
- Collapse whitespace, i.e multiple spaces or tabs back-to-back
- Wherever 3 or more new lines occur consecutively, replace them with 2 new lines

In particular, the gaps between paragraphs look like `\n \n \n` (i.e there is a space between new lines), so we'll need to account for that in the regex.

The new line processing is informed by the fact that LangChain's `RecursiveCharacterTextSplitter` -- which we'll use for chunking -- looks for `\n\n` as the plain-text representation of a paragraph break.

In [ ]:
import re


def clean_text(text):
    text = re.sub(r"[ \t]+", " ", text)  # collapse repeated spaces/tabs
    text = re.sub(r"[ \t]+\n", "\n", text)  # strip trailing space before a newline
    text = re.sub(
        r"\n{3,}", "\n\n", text
    )  # collapse blank-line runs to one paragraph break
    return text.strip()


assert clean_text(" Hello   World ") == "Hello World"
assert clean_text("Hello\n\n\nWorld") == "Hello\n\nWorld"
assert clean_text("Hello\n \n \nWorld") == "Hello\n\nWorld"

There are no line breaks in document 1, so let's test this cleaning function on document 2.

In [ ]:
loader = PyPDFLoader(doc2_path)
documents = loader.load()

doc2_page_content = documents[1].page_content
print(doc2_page_content)

In [ ]:
cleaned_content = clean_text(doc2_page_content)
print(cleaned_content)

As expected the multiple-line-breaks are reduced to single line breaks.

### Explore the extracted metadata

In [ ]:
for page_num, page in enumerate(documents):
    print(f"=== Page {page_num + 1} ===")
    print(page.metadata)
    print()

Looking at the metadata, there are a few things that would be useful to persist to the chunking step:
- `source`: currently this gives the whole file path, but we're just interested in the file name
- `page`: this will enable citations referencing the specific page number on which the chunk appears in the original document

We're not interested in metadata such as `producer`, `creator` and `moddate` as they don't relate to the document's content.

## Directory loader

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

In [ ]:
directory_path = "/Users/ashapatel/Documents/projects/rag_cc/documents/"

loader = DirectoryLoader(
    directory_path, glob="**/*.pdf", loader_cls=PyPDFLoader, show_progress=True
)
documents = loader.load()

print(f"Number of Documents: {len(documents)}")


for idx, document in enumerate(documents, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

This loads all documents in a given directory. This could be useful for my pipeline, but to start with I think I'll specify each document path separately.

# Classify page elements

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader

This loader is suited to PDFs with complex layouts, or scanned documents which require OCR to extract the text. It classifies page elements such as titles, tables, paragraphs etc.

In [ ]:
loader1 = UnstructuredPDFLoader(doc1_path)
documents1 = loader1.load()

print(f"Number of Documents: {len(documents1)}")

for idx, document in enumerate(documents1, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

For document 1 this loader seems to extract an inconsistent line length. It also puts multiple bullet points onto the same line, which seems strange.

In [ ]:
loader2 = UnstructuredPDFLoader(doc2_path)
documents2 = loader2.load()

print(f"Number of Documents: {len(documents2)}")

for idx, document in enumerate(documents2, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

For document 2, each section number is put on a separate line from the section heading, which didn't happen with PyPDFLoader.

Even if this loader did work better than simpler ones, we'd need to consider the extra processing time it takes.

## Future Work

#### Scaling to more documents

I am starting with a small dataset that I can manually inspect, but this wouldn't scale if I was working with 1000s of documents. In that case, the quality of the parse becomes more important in order to reduce the need for format-specific cleaning.


#### Golden test set

With a larger dataset I'd label a small set of samples with diverse layouts. Each time I make a change to the parsing/cleaning pipeline I could look at how this affects the samples in the golden data set.


#### Detecting anomalies

I could select features such as chunk length, lines with single characters etc and calculate the distribution of these features across all documents. I could then manually investigate anomalies.

The important thing to come back to is the effect of parsing & cleaning on retrieval. I'd have a golden dataset of question-answer pairs with associated relevant chunks in order to evaluate retrieval, and I could measure any changes that result from changes to parsing / cleaning.
